### Importação das bibliotecas e carregamento dos dados pré-processados

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

from src.preprocessing import load_processed_data
from src.evaluation import evaluate_model, save_results

# Carregando os conjuntos já pré-processados (One-Hot Encoding + SMOTE no treino),
# gerados pelo notebook 01 e compartilhados entre os modelos da equipe
X_train_res, X_val, X_test, y_train_res, y_val, y_test, preprocessador = load_processed_data('../data/processed')

print(f"Dimensão de X_train_res: {X_train_res.shape}")
print(f"Dimensão de X_val: {X_val.shape}")
print(f"Dimensão de X_test: {X_test.shape}")


### Treinamento do modelo base

In [ ]:
from xgboost import XGBClassifier

# Modelo com hiperparâmetros padrão, usado como referência antes da tunagem
modelo_base = XGBClassifier(
    n_estimators=200,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)
modelo_base.fit(X_train_res, y_train_res)


### Avaliação do modelo base no conjunto de validação

In [ ]:
metricas_base = evaluate_model(modelo_base, X_val, y_val, model_name='XGBoost (base)')


### Tunagem de hiperparâmetros

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Busca aleatória sobre a taxa de aprendizado, profundidade máxima, número de
# árvores e amostragem, usando AUC-ROC como métrica de seleção
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}

busca = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=8,
    scoring='roc_auc',
    cv=2,
    random_state=42,
    n_jobs=-1,
)

busca.fit(X_train_res, y_train_res)

print("Melhores hiperparâmetros:", busca.best_params_)
print(f"Melhor AUC-ROC (CV): {busca.best_score_:.4f}")

modelo_otimizado = busca.best_estimator_


### Avaliação do modelo otimizado no conjunto de validação

In [ ]:
metricas_val = evaluate_model(modelo_otimizado, X_val, y_val, model_name='XGBoost')


### Avaliação final no conjunto de teste interno

In [ ]:
metricas_teste = evaluate_model(modelo_otimizado, X_test, y_test, model_name='XGBoost')

# Salvando as métricas para a comparação entre modelos na Fase 3
save_results(metricas_teste, output_path='../results/resultados_modelos.csv')


### Salvando o modelo treinado

In [ ]:
import joblib
from pathlib import Path

# Modelo salvo para reuso, sem precisar repetir o treinamento e a tunagem
Path('../models').mkdir(parents=True, exist_ok=True)
joblib.dump(modelo_otimizado, '../models/modelo_xgboost.pkl')

print("Modelo salvo em models/modelo_xgboost.pkl")
